# Chapter 3 — One compile per signature

Step 3 builds the **two-tier cache** — and with it, Phase I's promise comes
due: the thesis is provable end-to-end **before any compiler exists**. The
"artifacts" in this chapter are fake strings; everything else — keys, guards,
generations, eviction, retirement, thread-safety — is the real machinery the
GPU pipelines will live in from ch09 on.

```
tier 1  SPECIALIZATION    (fp_head, arg_fp, backend_fp, generation) -> FastRecord
                   |  miss only
tier 2  ARTIFACT  (content_key, backend_token, flags) -> compiled artifact
                   |  miss only            (content-addressed, generation-FREE)
                   v
              lower -> rewrite -> render -> backend compile   (chapters 4-8)
```

| File | Counted lines / cap | What |
|---|---|---|
| `src/pdum/dsl/kernel/cache.py` | 147 / 150 | both tiers, futures, guards, LRU, retirement, `no_compile`, `explain_miss` |

Glossary terms settled: *specialization cache, artifact cache, generation, guard,
FastRecord (partially — its hit-path fields fill in ch08/ch09).*

In [1]:
import itertools

from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.cache import ArtifactCache, CompileForbidden, FastRecord, SpecializationCache, no_compile
from pdum.dsl.kernel.valuekind import fingerprint

cache = SpecializationCache()
serial = itertools.count(1)


def fake_gpu_compile(handle):
    # Stand-in for lower -> rewrite -> render -> backend.compile (ch05-ch09).
    return f"<pipeline #{next(serial)} for {handle.fntype!r}>"


def draw(handle, *args, backend=("fake-gpu", "rgba8unorm")):
    key = cache.key_for(handle, tuple(fingerprint(a) for a in args), backend)
    rec = cache.get_or_compile(key, lambda: FastRecord(artifact=fake_gpu_compile(handle)))
    return rec.artifact


def disk(cx, cy, radius):
    @jit(kind="fragment")
    def shader():
        return (cx, cy, radius)  # the body is irrelevant until ch07 — identity is what's real

    return shader


print(draw(disk(100.0, 100.0, 50.0)))
print(draw(disk(300.0, 200.0, 75.0)))
print(f"compiles={cache.compiles}  hits={cache.hits}")

<pipeline #1 for Fn<disk.<locals>.shader>(f64, f64, f64)>
<pipeline #1 for Fn<disk.<locals>.shader>(f64, f64, f64)>
compiles=1  hits=1


The key is four components, and **no captured value appears in any of them**:
`fp_head` is the Handle's precomputed `("H", code, env_fp)` digest (template
identity + capture *types*), then argument fingerprints, the backend token +
codegen-relevant params (M0's missing-format fault, cured structurally), and
the generation. Values ride in `Handle.env`, headed for uniform buffers —
never through here.

In [2]:
# The M0 acceptance test, on the new kernel, with no compiler in existence.
import math

with no_compile():  # from here on, ANY compile raises — the loop must be hot
    for k in range(300):
        t = k * 0.05
        draw(disk(320 + 180 * math.cos(t), 240 + 120 * math.sin(t), 70.0))

print(f"rebuilds=300  compiles={cache.compiles}  hits={cache.hits}")
assert cache.compiles == 1, "one compile, three hundred value-fresh frames"


rebuilds=300  compiles=1  hits=301


In [3]:
from pdum.dsl import viz

viz.install()
cache  # the cache renders its counters as pills — watch them across the chapter

In [4]:
# Perturb each key component; under no_compile() the miss RAISES, and the
# error NAMES the nearest entry's differing component. Key completeness is
# an experience, not a comment.
probes = {
    "a capture turns int": lambda: draw(disk(100, 100.0, 50.0)),
    "an argument appears": lambda: draw(disk(100.0, 100.0, 50.0), 1),
    "backend changes":     lambda: draw(disk(100.0, 100.0, 50.0), backend=("webgpu", "bgra8")),
}
for what, probe in probes.items():
    try:
        with no_compile():
            probe()
    except CompileForbidden as e:
        print(f"{what:22s} -> {e}")

a capture turns int    -> specialization-tier miss under no_compile(): nearest entry differs in: env_types
an argument appears    -> specialization-tier miss under no_compile(): nearest entry differs in: arg_types
backend changes        -> specialization-tier miss under no_compile(): nearest entry differs in: backend


In [5]:
# Live-coding hygiene: an EDIT retires the predecessor's entries (no leak),
# and bump_generation() is the sledgehammer that clears tier 1 wholesale.
from pdum.dsl.kernel.capture import make_handle

SRC_V1 = '''
def f(k):
    def g():
        return k
    return g
'''
SRC_V2 = SRC_V1.replace("return k", "return 2 * k")


def handle_from(src):
    ns = {}
    exec(compile(src, "<mymodule.py>", "exec"), ns)  # same def site both times
    return make_handle(ns["f"](7), "device")


draw(handle_from(SRC_V1))
print(f"v1 cached:      entries={len(cache)}  retirements={cache.retirements}")
draw(handle_from(SRC_V2))
print(f"after the edit: entries={len(cache)}  retirements={cache.retirements}  <- v1 retired, not leaked")

cache.bump_generation()
print(f"after bump:     entries={len(cache)}  generation={cache.generation}")
draw(disk(1.0, 2.0, 3.0))
print(f"same closure, new world: compiles={cache.compiles}")

v1 cached:      entries=2  retirements=0
after the edit: entries=2  retirements=1  <- v1 retired, not leaked
after bump:     entries=0  generation=1
same closure, new world: compiles=4


In [6]:
# Guards: a rebound dependency (think: frozen global) must never serve stale.
FROZEN = {"palette": object()}  # object identity stands in for a folded global


def draw_guarded(handle):
    key = cache.key_for(handle, (), ("fake-gpu",))
    rec = cache.get_or_compile(
        key,
        lambda: FastRecord(artifact=fake_gpu_compile(handle), guards=((FROZEN, "palette", FROZEN["palette"]),)),
    )
    return rec.artifact


a = draw_guarded(disk(9.0, 9.0, 9.0))
print("hit while the dependency is stable:", draw_guarded(disk(8.0, 8.0, 8.0)) is a)
FROZEN["palette"] = object()  # drift!
b = draw_guarded(disk(7.0, 7.0, 7.0))
print(f"refused + recompiled after drift:  {b is not a}   guard_misses={cache.guard_misses}")

hit while the dependency is stable: True
refused + recompiled after drift:  True   guard_misses=1


In [7]:
import threading

race, results = SpecializationCache(), []
barrier = threading.Barrier(8)


def worker():
    barrier.wait()
    key = race.key_for(disk(1.0, 1.0, 1.0))
    rec = race.get_or_compile(key, lambda: FastRecord(artifact=f"<built by {threading.current_thread().name}>"))
    results.append(rec.artifact)


threads = [threading.Thread(target=worker) for _ in range(8)]
[t.start() for t in threads]
[t.join() for t in threads]
print(f"8 threads raced one key: compiles={race.compiles}, distinct artifacts={len(set(results))}")

8 threads raced one key: compiles=1, distinct artifacts=1


In [8]:
# Tier 2 is CONTENT-addressed and generation-free: once lowering exists,
# two templates producing identical IR share one compiled artifact.
art, n = ArtifactCache(), itertools.count(1)


def build():
    return f"<compiled blob #{next(n)}>"


print(art.get_or_compile(("sha256:9f3a...", "wgsl", ()), build))
print(art.get_or_compile(("sha256:9f3a...", "wgsl", ()), build), " <- same content: same blob")
print(art.get_or_compile(("sha256:9f3a...", "c", ()), build), "    <- same content, other backend")
print(f"artifact-tier compiles={art.compiles}")

<compiled blob #1>
<compiled blob #1>  <- same content: same blob
<compiled blob #2>     <- same content, other backend
artifact-tier compiles=2


## The two-tier law, experienced

| When this changes… | …this misses |
|---|---|
| a captured **value** (the hot loop) | **nothing** — pure hit |
| a capture or argument **type** | specialization tier |
| the function **body** (edit + rerun) | specialization tier (+ predecessor retired) |
| a **backend** or codegen flag | specialization tier (+ artifact tier on first sight) |
| a frozen **dependency** (guard drift) | specialization tier, by refusal — never stale |
| `bump_generation()` | all of tier 1; tier 2 untouched (content-addressed) |

**Phase I is complete: the thesis is now a passing test suite on the new
kernel — and no compiler exists yet.**

## What we can't do yet

- `FastRecord.extract/plan/staging/launch` are `None` — the precompiled hit
  path fills in at ch08/ch09; today the "artifact" is a string.
- Guards are synthetic — real dependency tags come from `classify_names`
  at lowering (ch07).
- Calling a Handle still does nothing; `draw()` here is the shape of the
  ch09 dispatch, written by hand.

**Budget after step 3:** kernel 378 / 1150 counted lines.

**Next:** ch04 — pipelines are values: the blessed combinator layer,
attached with zero kernel edits.